<div dir="rtl">

# 🔗 02 - LCEL Syntax & Output Parsers

## ما هي لغة التعبير LCEL ومحللات المخرجات؟
- **LCEL (LangChain Expression Language)**: المعمارية الحديثة الموحدة لربط مكونات LangChain باستخدام مشغل الأنابيب `|` (Pipe Operator).
- **StrOutputParser**: محلل مخرجات يلتقط كائن `AIMessage` ويستخرج منه النص الصافي (`response.content`) تلقائياً.
- **RunnablePassthrough**: مكون لتمرير المدخلات كما هي للخطوات التالية بالتوازي دون أي تعديل.

</div>

<div dir="rtl">

### ⚙️ تهيئة البيئة والنموذج
تحميل المتغيرات البيئية وتهيئة كائن `ChatGroq`.

</div>

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# تحميل البيئة
load_dotenv(find_dotenv())

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)
print("✅ تم تهيئة البيئة ونموذج ChatGroq بنجاح!")


<div dir="rtl">

### 1️⃣ السلسلة الأساسية باستخدام مشغل LCEL (`prompt | llm | parser`)
نقوم بربط القالب مع النموذج مع محلل المخرجات في خط أنابيب واحد انسيابي وأنيق.

</div>

In [ ]:
# 1. تعريف القالب
prompt = ChatPromptTemplate.from_messages([
    ("system", "أنت مساعد ذكي متخصص في تبسيط المفاهيم العلمية."),
    ("human", "اشرح مفهوم {concept} للأطفال في جملتين فقط.")
])

# 2. بناء السلسلة باستخدام مشغل LCEL |
chain = prompt | llm | StrOutputParser()

# 3. تنفيذ السلسلة
result = chain.invoke({"concept": "الجاذبية الأرضية"})
print("📝 مخرجات السلسلة (نص نقي مباشرة):")
print(result)
print(f"\nنوع الناتج: {type(result)}")


<div dir="rtl">

### 2️⃣ تمرير المدخلات المباشرة عبر `RunnablePassthrough`
عندما يكون المدخل مجرد نص مفرد (String)، يساعدنا `RunnablePassthrough` على تحويله إلى قاموس بالمتغيرات المطلوبة للقالب بسلاسة.

</div>

In [ ]:
qa_prompt = ChatPromptTemplate.from_template(
    "أجب عن السؤال التالي بدقة وإيجاز:\nالسؤال: {question}\nالإجابة:"
)

# سلسلة تقبل نص السؤال مباشرة كـ string
qa_chain = (
    {"question": RunnablePassthrough()}
    | qa_prompt
    | llm
    | StrOutputParser()
)

answer = qa_chain.invoke("ما هي عاصمة اليابان وما هي عملتها الرسمية؟")
print(answer)


<div dir="rtl">

### 3️⃣ التنفيذ المتوازي والتفرع عبر `RunnableParallel`
يتيح `RunnableParallel` تنفيذ عدة مهام وسلاسل معاً في نفس الوقت، مثل تلخيص نص وترجمته في خطوة واحدة.

</div>

In [ ]:
# قوالب لمهام متعددة
summary_prompt = ChatPromptTemplate.from_template("Summarize this text in 5 words:\n{text}")
translation_prompt = ChatPromptTemplate.from_template("Translate this text to French:\n{text}")

# بناء سلسلتين منفصلتين
summary_chain = summary_prompt | llm | StrOutputParser()
translation_chain = translation_prompt | llm | StrOutputParser()

# الجمع بالتوازي
parallel_chain = RunnableParallel({
    "original": RunnablePassthrough(),
    "summary": summary_chain,
    "french_translation": translation_chain
})

sample_text = "LangChain is a powerful framework designed to simplify the creation of applications using large language models."
results = parallel_chain.invoke({"text": sample_text})

print("📊 نتائج المعالجة المتوازية:")
for k, v in results.items():
    print(f"🔹 {k}: {v}")


<div dir="rtl">

## 💡 خلاصة وخاتمة (Summary & Next Steps)
- تعلمنا كيفية استخدام مشغل الربط `|` لبناء سلاسل LCEL احترافية.
- استخدمنا `StrOutputParser` لاستخراج النصوص النظيفة مباشرة.
- جربنا `RunnablePassthrough` لتمرير النصوص و `RunnableParallel` للمعالجة المتعددة المتزامنة.
- **الخطوة التالية**: الانتقال إلى كراس `03_rag_chains_with_retrievers.ipynb` لربط مستودعات المتجهات (Retrievers) بسلاسل الـ LCEL.

</div>